<a href="https://colab.research.google.com/github/SanaFathimaKS/ML-LAB1/blob/main/Exp_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [3]:
# Load Titanic dataset
df = pd.read_csv("titanic.csv")

In [4]:
# Select features
features = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]

X = df[features]
y = df["Survived"]


In [5]:
# Feature types
numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]


In [6]:
# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

In [8]:
# Base Decision Tree
base_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

In [9]:
# Bagging
bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    random_state=42
)

bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)

In [17]:


# Re-define robust preprocessor locally to ensure NaNs are handled
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

local_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# AdaBoost
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    ),
    n_estimators=50,
    random_state=42
)

boosting_pipeline = Pipeline([
    ("preprocessor", local_preprocessor),
    ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

In [18]:
# Evaluation function
def evaluate_model(name, y_true, y_pred):
    print("\n", name)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Precision:",
          round(precision_score(y_true, y_pred), 4))

    print("Recall:",
          round(recall_score(y_true, y_pred), 4))

    print("F1-score:",
          round(f1_score(y_true, y_pred), 4))

In [19]:
# Evaluate
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)


 Bagging
Accuracy: 0.7989
Precision: 0.8667
Recall: 0.5652
F1-score: 0.6842

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774
